#### Step 4: Tokenizer Evaluation

**Why:** we trained our own tokenizer (`AxisQuant/IndicPrayog-tokenizer-64k`) for English, Hindi, and Marathi. Before finalizing the data mixture, we need to know how efficiently it encodes each language. Devanagari script (Hindi, Marathi) usually needs more tokens per character than Latin script (English), because a single Devanagari character often maps to more than one byte/subword unit. If we do not measure this, we could accidentally under budget tokens for Hindi/Marathi when building the training mixture.

**What we are solving:** build a small evaluation corpus covering English, Hindi, Marathi, and a few tricky categories (code, numbers, URLs, punctuation, names, mixed language text), run the tokenizer on each, and measure:

```
chars / token   (compression ratio, higher is more efficient)
tokens / word
```

for each category, so we can see exactly which languages/categories cost more tokens and plan the vocabulary and data mixture accordingly.

In [3]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

DATASET_ROOT = Path("../../../../dataset").resolve()
STANDARDIZED_ROOT = DATASET_ROOT / "standardized"
TOKENIZER_REPO = "AxisQuant/IndicPrayog-tokenizer-64k"
SAMPLE_CHARS_PER_LANGUAGE = 200_000

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO)
print(f"vocab size: {tokenizer.vocab_size}")

vocab size: 64000


#### Build the evaluation corpus

For English, Hindi, and Marathi we pull real text straight from our Step 1 standardized shards, up to `SAMPLE_CHARS_PER_LANGUAGE` characters each. For the tricky categories (code, numbers, URLs, punctuation, names, mixed language) there is no dedicated dataset, so we use small representative snippets instead.

In [4]:
def sample_language_text(language, max_chars):
    shards = sorted((STANDARDIZED_ROOT / language).glob("*/*.parquet"))
    chunks = []
    total = 0
    for shard in shards:
        df = pd.read_parquet(shard, columns=["text"])
        for text in df["text"]:
            chunks.append(text)
            total += len(text)
            if total >= max_chars:
                return "\n".join(chunks)[:max_chars]
    return "\n".join(chunks)

english_text = sample_language_text("en", SAMPLE_CHARS_PER_LANGUAGE)
hindi_text = sample_language_text("hi", SAMPLE_CHARS_PER_LANGUAGE)
marathi_text = sample_language_text("mr", SAMPLE_CHARS_PER_LANGUAGE)

print(f"english: {len(english_text)} chars")
print(f"hindi: {len(hindi_text)} chars")
print(f"marathi: {len(marathi_text)} chars")

english: 200000 chars
hindi: 200000 chars
marathi: 200000 chars


In [5]:
code_text = "\n".join([
    "def compute_average(values):",
    "    total = sum(values)",
    "    return total / len(values) if values else 0.0",
    "",
    "class DataLoader:",
    "    def __init__(self, batch_size=32, shuffle=True):",
    "        self.batch_size = batch_size",
    "        self.shuffle = shuffle",
    "",
    "    def __iter__(self):",
    "        for i in range(0, len(self.data), self.batch_size):",
    "            yield self.data[i:i + self.batch_size]",
] * 40)

numbers_text = " ".join([
    "3.14159", "42", "1000000", "-273.15", "0x1F4A9", "2026-08-27", "93.5%",
    "1,234,567.89", "1e-9", "98765432101234", "12:45:30", "+91 98765 43210",
] * 60)

urls_text = " ".join([
    "https://huggingface.co/AxisQuant/IndicPrayog-tokenizer-64k",
    "https://en.wikipedia.org/wiki/Devanagari",
    "https://github.com/PrashantTakale369/IndicPrayog-0.5B-A50M/pull/6",
    "http://example.com/path/to/page?query=value&lang=hi#section",
    "www.data.gov.in/resource/dataset",
] * 40)

punctuation_text = (
    "Hello, world! How are you? I'm fine... \"Really,\" she said; "
    "नमस्ते, आप कैसे हैं? मैं ठीक हूँ... \u201cसच में,\u201d उसने कहा; "
    "नमस्कार, तुम्ही कसे आहात? मी ठीक आहे... [ब्रॅकेट्स] {कर्ली} <कोन>! "
) * 80

names_text = " ".join([
    "Prashant Takale", "Amitabh Bachchan", "Sachin Tendulkar", "Lata Mangeshkar",
    "\u0930\u0935\u0940\u0902\u0926\u094d\u0930 \u091c\u0926\u0947\u091c\u093e", "\u092b\u0941\u0932\u0947 \u0936\u0947\u0902\u0926\u0941\u0930\u0915\u0930",
    "\u0938\u094c\u0930\u0935 \u0917\u093e\u0902\u0917\u0941\u0932\u0940", "\u091c\u094d\u092f\u094b\u0924\u093f\u0930\u093e\u0935 \u092b\u0941\u0932\u0947",
    "Narendra Modi", "A.P.J. Abdul Kalam",
] * 40)

mixed_text = (
    "आज मैंने एक नया Python script लिखा जो dataset को process करता है, "
    "त्यानंतर मी GitHub वर push केले आणि PR उघडली. "
    "The model achieved 94.2% accuracy on the validation set, म्हणजे खूप छान result आहे. "
    "कृपया इस URL को चेक करें: https://huggingface.co/AxisQuant/IndicPrayog-tokenizer-64k "
) * 60

print("synthetic samples ready")

synthetic samples ready


In [6]:
corpus = {
    "english": english_text,
    "hindi": hindi_text,
    "marathi": marathi_text,
    "code": code_text,
    "numbers": numbers_text,
    "urls": urls_text,
    "punctuation": punctuation_text,
    "names": names_text,
    "mixed_language": mixed_text,
}

#### Measure tokenizer efficiency

For each category: encode the text, count tokens, and compute chars/token (compression ratio) and tokens/word.

In [7]:
def measure(text):
    token_ids = tokenizer.encode(text)
    num_tokens = len(token_ids)
    num_chars = len(text)
    num_words = len(text.split())
    return {
        "chars": num_chars,
        "tokens": num_tokens,
        "chars_per_token": num_chars / num_tokens if num_tokens else 0.0,
        "tokens_per_word": num_tokens / num_words if num_words else 0.0,
    }

In [8]:
results = []
for category, text in corpus.items():
    stats = measure(text)
    stats["category"] = category
    results.append(stats)

report = pd.DataFrame(results)[["category", "chars", "tokens", "chars_per_token", "tokens_per_word"]]
report = report.sort_values("chars_per_token")
report_path = Path("tokenizer_eval_report.csv")
report.to_csv(report_path, index=False)
print(f"saved: {report_path.resolve()}")
report

saved: /home/contributor/users/prashant.takale/github_v2/IndicPrayog-0.5B-A50M/dataset_preparation/dataset_complete_pipeline/4_tokenization/tokenizer_eval_report.csv


,category,chars,tokens,chars_per_token,tokens_per_word
4,numbers,6599,3180,2.075157,3.785714
3,code,15159,7120,2.129073,4.810811
6,punctuation,14560,5041,2.888316,2.032661
5,urls,10359,3520,2.942898,17.600000
7,names,6199,1720,3.604070,2.047619
8,mixed_language,16860,4441,3.796442,1.721318
1,hindi,200000,50150,3.988036,1.288772
2,marathi,200000,45086,4.435967,1.547858
0,english,200000,41804,4.784231,1.287346


#### How to read this

Lower `chars_per_token` means the tokenizer needs more tokens to represent the same amount of text, i.e. that category is more expensive. If Hindi and Marathi come out noticeably lower than English, that confirms the vocabulary/data mixture needs to give Hindi and Marathi a larger token budget (or oversample them) so the model sees a comparable amount of real content per training step, not just per document.

#### Full dataset token count

The section above only measures efficiency on a small sample. This section tokenizes every row of every standardized shard (all languages, all sources) to get the real total token count of the entire dataset, this is what you need to know how many training tokens you actually have. This runs on CPU only (the tokenizer has no GPU backend). A real benchmark on one shard measured about 4,160 docs/sec, so the full ~29.5M rows should take roughly 2 hours.

In [9]:
from tqdm.auto import tqdm

BATCH_SIZE = 1000

all_shards = sorted(STANDARDIZED_ROOT.glob("*/*/*.parquet"))
print(f"found {len(all_shards)} shard(s) to tokenize")

full_count_rows = []
shard_bar = tqdm(all_shards, desc="shards", unit="shard")
for shard in shard_bar:
    language = shard.parent.parent.name
    source = shard.parent.name
    shard_bar.set_postfix(current=f"{language}/{source}/{shard.name}")

    df = pd.read_parquet(shard, columns=["text"])
    texts = df["text"].tolist()

    shard_tokens = 0
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="rows", unit="batch", leave=False):
        batch = texts[start:start + BATCH_SIZE]
        encoded = tokenizer(batch)["input_ids"]
        shard_tokens += sum(len(ids) for ids in encoded)

    shard_chars = df["text"].str.len().sum()

    full_count_rows.append({
        "language": language,
        "source": source,
        "shard": shard.name,
        "rows": len(df),
        "chars": shard_chars,
        "tokens": shard_tokens,
    })

full_counts = pd.DataFrame(full_count_rows)
full_counts_path = Path("full_dataset_token_counts.csv")
full_counts.to_csv(full_counts_path, index=False)
print(f"saved: {full_counts_path.resolve()}")

found 149 shard(s) to tokenize


shards: 100%|██████████| 149/149 [2:30:30<00:00, 60.61s/shard, current=mr/wikipedia/train-00000-of-00001.parquet]

saved: /home/contributor/users/prashant.takale/github_v2/IndicPrayog-0.5B-A50M/dataset_preparation/dataset_complete_pipeline/4_tokenization/full_dataset_token_counts.csv


In [10]:
by_language = full_counts.groupby("language")[["rows", "chars", "tokens"]].sum()
by_language["chars_per_token"] = by_language["chars"] / by_language["tokens"]

print(f"TOTAL rows: {full_counts['rows'].sum():,}")
print(f"TOTAL chars: {full_counts['chars'].sum():,}")
print(f"TOTAL tokens: {full_counts['tokens'].sum():,}")
print()
by_language

TOTAL rows: 29,456,220
TOTAL chars: 78,373,316,404
TOTAL tokens: 18,412,346,218



,rows,chars,tokens,chars_per_token
language,,,,
en,5912445,27048827002,5852291443,4.621921
hi,17584025,38509470798,9713056055,3.964712
mr,5959750,12815018604,2846998720,4.501238
